# Prometheus Geodynamic Model Training on Kaggle TPU

This notebook is designed to train volcanic and seismic prediction models using Kaggle's TPU resources.

## Setup Instructions
1. Upload this notebook to Kaggle
2. Enable TPU accelerator in notebook settings
3. For remote Jupyter access, run the connection setup cell below

## 1. Jupyter Server Setup for Remote IDE Connection

In [ ]:
# Install required packages for remote connection
!pip install -q jupyter_http_over_ws
!jupyter serverextension enable --py jupyter_http_over_ws

# Generate Jupyter config
!jupyter notebook --generate-config

# Set up password (you'll be prompted to enter one)
from notebook.auth import passwd
password = passwd()
print(f"Password hash: {password}")

# Configure Jupyter for remote access
import json
config = {
    "NotebookApp": {
        "ip": "0.0.0.0",
        "port": 8888,
        "allow_origin": "*",
        "allow_remote_access": True,
        "password": password
    }
}

with open('/root/.jupyter/jupyter_notebook_config.json', 'w') as f:
    json.dump(config, f)

print("Jupyter server configured for remote access")

In [ ]:
# Start Jupyter server in background (for Kaggle environment)
import subprocess
import time

# Start server
process = subprocess.Popen(['jupyter', 'notebook', '--no-browser'], 
                          stdout=subprocess.PIPE, 
                          stderr=subprocess.PIPE)

time.sleep(5)
print("Jupyter server started. Use ngrok or similar to expose port 8888 for external access.")

## 2. TPU Setup and Configuration

In [ ]:
import os
import sys
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Add current directory to path to import api_config
sys.path.append('/kaggle/working')

# Import API configuration if uploaded
try:
    from api_config import APICredentials, APIEndpoints, get_auth_headers
    print("API configuration loaded successfully")
except ImportError:
    print("Warning: api_config.py not found. Upload it to use real data fetching.")

# Detect TPU
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print(f'Running on TPU {tpu.master()}')
except ValueError:
    tpu = None
    print('TPU not found, falling back to GPU/CPU')

if tpu:
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
else:
    # Default strategy for GPU/CPU
    strategy = tf.distribute.get_strategy()

print(f"Number of replicas: {strategy.num_replicas_in_sync}")

## 3. Data Loading Functions

In [ ]:
def fetch_asf_hyp3_insar_data():
    """
    Fetch InSAR processed data from ASF HyP3 API
    """
    try:
        # Check if we have NASA EarthData credentials
        creds = APICredentials()
        if creds.EARTHDATA_TOKEN:
            headers = {
                "Authorization": f"Bearer {creds.EARTHDATA_TOKEN}",
                "Content-Type": "application/json"
            }
            
            # Search for recent Sentinel-1 scenes over volcanic regions
            # Example: Mount St. Helens
            search_params = {
                "platform": "Sentinel-1A,Sentinel-1B",
                "processingLevel": "SLC",
                "beamMode": "IW",
                "polarization": "VV+VH",
                "start": (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d'),
                "end": datetime.now().strftime('%Y-%m-%d'),
                "bbox": "-122.3,-46.1,-122.1,46.3",  # Mount St. Helens
                "output": "json"
            }
            
            # Search ASF for available scenes
            response = requests.get(
                "https://api.daac.asf.alaska.edu/services/search/param",
                params=search_params,
                headers=headers
            )
            
            if response.status_code == 200:
                scenes = response.json()
                print(f"Found {len(scenes)} Sentinel-1 scenes for InSAR processing")
                
                # For training, we'll simulate InSAR displacement data
                # Real implementation would submit jobs to HyP3 and download results
                return create_synthetic_insar_data(len(scenes))
            
    except Exception as e:
        print(f"Error fetching ASF data: {e}")
    
    return create_synthetic_insar_data()

def create_synthetic_insar_data(n_scenes=100):
    """
    Create synthetic InSAR displacement data for training
    In reality, this would be interferogram data from HyP3 processing
    """
    # InSAR data typically includes:
    # - Displacement maps (2D arrays)
    # - Coherence maps (quality indicator)
    # - Temporal baselines
    
    n_samples = min(n_scenes * 10, 5000)  # Multiple samples per scene
    img_size = 128  # Smaller patches from full interferograms
    
    # Displacement data (in mm)
    displacement = np.random.randn(n_samples, img_size, img_size, 1).astype(np.float32) * 50
    
    # Coherence data (0-1)
    coherence = np.random.beta(5, 2, (n_samples, img_size, img_size, 1)).astype(np.float32)
    
    # Combine displacement and coherence
    X_insar = np.concatenate([displacement, coherence], axis=-1)
    
    # Labels: detect significant deformation (>20mm)
    y_insar = (np.abs(displacement.mean(axis=(1,2,3))) > 20).astype(np.float32)
    
    print(f"Generated {n_samples} InSAR training samples")
    return X_insar, y_insar

def create_insar_cnn_model(input_shape, n_classes=2):
    """
    CNN model for InSAR deformation detection
    """
    with strategy.scope():
        model = keras.Sequential([
            layers.Input(shape=input_shape),
            
            # Feature extraction
            layers.Conv2D(32, 3, padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.Conv2D(32, 3, padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2),
            
            layers.Conv2D(64, 3, padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.Conv2D(64, 3, padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2),
            
            layers.Conv2D(128, 3, padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.Conv2D(128, 3, padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.GlobalAveragePooling2D(),
            
            # Classification head
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(n_classes, activation='softmax' if n_classes > 2 else 'sigmoid')
        ])
        
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
            loss='sparse_categorical_crossentropy' if n_classes > 2 else 'binary_crossentropy',
            metrics=['accuracy', keras.metrics.AUC()]
        )
    
    return model

def load_insar_data():
    """Load InSAR data - real from ASF if possible, synthetic as fallback"""
    print("Attempting to load InSAR data from ASF HyP3...")
    return fetch_asf_hyp3_insar_data()

## 3. InSAR Data Loading Functions

In [ ]:
# Configuration
BATCH_SIZE = 128 * strategy.num_replicas_in_sync
BUFFER_SIZE = 10000
EPOCHS = 50
LEARNING_RATE = 0.001

# Import required libraries for data fetching
import requests
import json
from datetime import datetime, timedelta

def fetch_real_seismic_data():
    """
    Fetch real seismic data from USGS API
    """
    try:
        # Use USGS API to fetch recent earthquake data
        end_date = datetime.now()
        start_date = end_date - timedelta(days=30)
        
        params = {
            "format": "geojson",
            "starttime": start_date.isoformat(),
            "endtime": end_date.isoformat(),
            "minmagnitude": 3.0,
            "limit": 1000
        }
        
        # Try to use API key if available
        headers = {}
        try:
            headers = get_auth_headers("usgs")
        except:
            pass
        
        response = requests.get(
            "https://earthquake.usgs.gov/fdsnws/event/1/query",
            params=params,
            headers=headers
        )
        
        if response.status_code == 200:
            data = response.json()
            earthquakes = []
            
            for feature in data['features']:
                props = feature['properties']
                coords = feature['geometry']['coordinates']
                
                # Extract features for training
                earthquakes.append([
                    coords[1],  # latitude
                    coords[0],  # longitude
                    coords[2],  # depth
                    props['mag'],  # magnitude
                    props.get('gap', 0),  # azimuthal gap
                    props.get('dmin', 0),  # distance to nearest station
                    props.get('rms', 0),  # root mean square
                    props.get('nst', 0)  # number of stations
                ])
            
            # Convert to numpy array
            X = np.array(earthquakes, dtype=np.float32)
            
            # Create labels based on magnitude (example: predict if M > 4.5)
            y = (X[:, 3] > 4.5).astype(np.float32)
            
            # Reshape for LSTM (add time dimension)
            # Group by location bins to create sequences
            n_samples = len(X) // 10
            X_seq = X[:n_samples*10].reshape(n_samples, 10, X.shape[1])
            y_seq = y[:n_samples*10].reshape(n_samples, 10).max(axis=1)
            
            print(f"Loaded {n_samples} seismic sequences from USGS")
            return X_seq, y_seq
            
    except Exception as e:
        print(f"Error fetching real data: {e}")
        print("Falling back to synthetic data")
    
    # Fallback to synthetic data
    return load_synthetic_seismic_data()

def load_synthetic_seismic_data():
    """
    Generate synthetic seismic data for demonstration
    """
    sequence_length = 100
    n_features = 8  # lat, lon, depth, mag, gap, dmin, rms, nst
    n_samples = 10000
    
    X = np.random.randn(n_samples, sequence_length, n_features).astype(np.float32)
    y = np.random.randint(0, 2, (n_samples,)).astype(np.float32)
    
    return X, y

def fetch_real_thermal_data():
    """
    Fetch metadata for thermal imagery from Sentinel Hub
    """
    try:
        # Check if we have Sentinel Hub credentials
        creds = APICredentials()
        if creds.COPERNICUS_CLIENT_ID:
            # Get token
            token_response = requests.post(
                "https://services.sentinel-hub.com/oauth/token",
                data={
                    "grant_type": "client_credentials",
                    "client_id": creds.COPERNICUS_CLIENT_ID,
                    "client_secret": creds.COPERNICUS_CLIENT_SECRET
                }
            )
            
            if token_response.status_code == 200:
                token = token_response.json()["access_token"]
                
                # Search for thermal data
                headers = {
                    "Authorization": f"Bearer {token}",
                    "Content-Type": "application/json"
                }
                
                # Example: Mount Etna bbox
                search_request = {
                    "bbox": [14.9, 37.7, 15.1, 37.8],
                    "datetime": f"{(datetime.now() - timedelta(days=30)).isoformat()}Z/{datetime.now().isoformat()}Z",
                    "collections": ["sentinel-3-slstr"],
                    "limit": 50
                }
                
                response = requests.post(
                    "https://services.sentinel-hub.com/api/v1/catalog/search",
                    json=search_request,
                    headers=headers
                )
                
                if response.status_code == 200:
                    scenes = response.json().get('features', [])
                    print(f"Found {len(scenes)} thermal scenes metadata")
                    # Note: Actual image download would require additional processing
                    
    except Exception as e:
        print(f"Error fetching thermal metadata: {e}")
    
    # For training, use synthetic data
    return load_synthetic_thermal_data()

def load_synthetic_thermal_data():
    """
    Generate synthetic thermal imagery data
    """
    img_height, img_width = 224, 224
    n_channels = 3
    n_samples = 5000
    
    X = np.random.rand(n_samples, img_height, img_width, n_channels).astype(np.float32)
    # Multi-class: 0=no activity, 1=low, 2=moderate, 3=high, 4=extreme
    y = np.random.randint(0, 5, (n_samples,))
    
    return X, y

# Wrapper functions that try real data first
def load_seismic_data():
    """Load seismic data - real if possible, synthetic as fallback"""
    print("Attempting to load real seismic data from USGS...")
    return fetch_real_seismic_data()

def load_volcanic_thermal_data():
    """Load volcanic thermal data - metadata from Sentinel Hub, synthetic for training"""
    print("Loading volcanic thermal data...")
    return fetch_real_thermal_data()

## 4. Model Architectures

In [ ]:
def create_seismic_lstm_model(input_shape, n_classes=2):
    """
    LSTM model for seismic time series classification
    """
    with strategy.scope():
        model = keras.Sequential([
            layers.Input(shape=input_shape),
            layers.LSTM(128, return_sequences=True),
            layers.BatchNormalization(),
            layers.LSTM(64, return_sequences=True),
            layers.BatchNormalization(),
            layers.LSTM(32),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(n_classes, activation='softmax' if n_classes > 2 else 'sigmoid')
        ])
        
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
            loss='sparse_categorical_crossentropy' if n_classes > 2 else 'binary_crossentropy',
            metrics=['accuracy']
        )
    
    return model

def create_volcanic_cnn_model(input_shape, n_classes=5):
    """
    CNN model for volcanic thermal image classification
    """
    with strategy.scope():
        # Using EfficientNet as backbone
        base_model = tf.keras.applications.EfficientNetB0(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
        base_model.trainable = False  # Freeze base model initially
        
        model = keras.Sequential([
            base_model,
            layers.GlobalAveragePooling2D(),
            layers.BatchNormalization(),
            layers.Dense(512, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(n_classes, activation='softmax')
        ])
        
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3)]
        )
    
    return model

def create_comprehensive_fusion_model(seismic_shape, thermal_shape, insar_shape, n_classes=4):
    """
    Advanced multimodal model combining seismic, thermal, and InSAR data
    """
    with strategy.scope():
        # Seismic branch (LSTM)
        seismic_input = layers.Input(shape=seismic_shape, name='seismic_input')
        seismic_lstm = layers.LSTM(128, return_sequences=True)(seismic_input)
        seismic_lstm = layers.LSTM(64)(seismic_lstm)
        seismic_features = layers.Dense(128, activation='relu')(seismic_lstm)
        seismic_features = layers.Dropout(0.3)(seismic_features)
        
        # Thermal branch (CNN)
        thermal_input = layers.Input(shape=thermal_shape, name='thermal_input')
        thermal_base = tf.keras.applications.EfficientNetB0(
            input_shape=thermal_shape,
            include_top=False,
            weights='imagenet'
        )
        thermal_base.trainable = False
        thermal_features = thermal_base(thermal_input)
        thermal_features = layers.GlobalAveragePooling2D()(thermal_features)
        thermal_features = layers.Dense(128, activation='relu')(thermal_features)
        thermal_features = layers.Dropout(0.3)(thermal_features)
        
        # InSAR branch (CNN)
        insar_input = layers.Input(shape=insar_shape, name='insar_input')
        insar_conv = layers.Conv2D(32, 3, padding='same', activation='relu')(insar_input)
        insar_conv = layers.BatchNormalization()(insar_conv)
        insar_conv = layers.MaxPooling2D(2)(insar_conv)
        insar_conv = layers.Conv2D(64, 3, padding='same', activation='relu')(insar_conv)
        insar_conv = layers.BatchNormalization()(insar_conv)
        insar_conv = layers.MaxPooling2D(2)(insar_conv)
        insar_conv = layers.Conv2D(128, 3, padding='same', activation='relu')(insar_conv)
        insar_conv = layers.GlobalAveragePooling2D()(insar_conv)
        insar_features = layers.Dense(128, activation='relu')(insar_conv)
        insar_features = layers.Dropout(0.3)(insar_features)
        
        # Multi-level fusion
        # Early fusion of seismic and InSAR (both ground-based measurements)
        ground_fusion = layers.concatenate([seismic_features, insar_features])
        ground_fusion = layers.Dense(128, activation='relu')(ground_fusion)
        ground_fusion = layers.BatchNormalization()(ground_fusion)
        
        # Late fusion with thermal
        combined = layers.concatenate([ground_fusion, thermal_features])
        
        # Deep fusion layers
        fusion = layers.Dense(256, activation='relu')(combined)
        fusion = layers.BatchNormalization()(fusion)
        fusion = layers.Dropout(0.5)(fusion)
        fusion = layers.Dense(128, activation='relu')(fusion)
        fusion = layers.BatchNormalization()(fusion)
        fusion = layers.Dropout(0.3)(fusion)
        
        # Output layer with attention mechanism
        attention = layers.Dense(128, activation='tanh')(fusion)
        attention = layers.Dense(128, activation='softmax')(attention)
        fusion = layers.multiply([fusion, attention])
        
        output = layers.Dense(n_classes, activation='softmax')(fusion)
        
        model = keras.Model(
            inputs=[seismic_input, thermal_input, insar_input],
            outputs=output
        )
        
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2)]
        )
    
    return model

## 5. Training Pipeline

In [ ]:
def create_dataset(X, y, batch_size, shuffle=True):
    """
    Create tf.data.Dataset optimized for TPU
    """
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    
    if shuffle:
        dataset = dataset.shuffle(BUFFER_SIZE)
    
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

def train_model(model, train_dataset, val_dataset, model_name):
    """
    Train model with callbacks
    """
    # Callbacks
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            f'{model_name}_best.h5',
            monitor='val_accuracy',
            save_best_only=True,
            mode='max'
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7
        ),
        keras.callbacks.TensorBoard(
            log_dir=f'./logs/{model_name}',
            histogram_freq=1
        )
    ]
    
    # Train
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=val_dataset,
        callbacks=callbacks,
        verbose=1
    )
    
    return history

def plot_training_history(history, model_name):
    """
    Plot training metrics
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(f'{model_name} - Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)
    
    # Loss
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(f'{model_name} - Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(f'{model_name}_training_history.png')
    plt.show()

## 6. Train Seismic Model

In [ ]:
# Load seismic data
print("Loading seismic data...")
X_seismic, y_seismic = load_seismic_data()

# Adjust input shape based on actual data
input_shape = X_seismic.shape[1:]  # (sequence_length, n_features)
print(f"Seismic data shape: {X_seismic.shape}")
print(f"Input shape for model: {input_shape}")

# Split data
split_idx = int(0.8 * len(X_seismic))
X_train, X_val = X_seismic[:split_idx], X_seismic[split_idx:]
y_train, y_val = y_seismic[:split_idx], y_seismic[split_idx:]

# Create datasets
train_dataset = create_dataset(X_train, y_train, BATCH_SIZE)
val_dataset = create_dataset(X_val, y_val, BATCH_SIZE, shuffle=False)

# Create and train model
print("Creating seismic LSTM model...")
seismic_model = create_seismic_lstm_model(input_shape=input_shape, n_classes=2)
seismic_model.summary()

print("\nTraining seismic model...")
seismic_history = train_model(seismic_model, train_dataset, val_dataset, 'seismic_lstm')

# Plot results
plot_training_history(seismic_history, 'Seismic LSTM')

## 7. Train Volcanic Thermal Model

In [ ]:
# Load volcanic thermal data
print("Loading volcanic thermal data...")
X_thermal, y_thermal = load_volcanic_thermal_data()

# Preprocess images
X_thermal = tf.keras.applications.efficientnet.preprocess_input(X_thermal)

# Split data
split_idx = int(0.8 * len(X_thermal))
X_train, X_val = X_thermal[:split_idx], X_thermal[split_idx:]
y_train, y_val = y_thermal[:split_idx], y_thermal[split_idx:]

# Create datasets
train_dataset = create_dataset(X_train, y_train, BATCH_SIZE)
val_dataset = create_dataset(X_val, y_val, BATCH_SIZE, shuffle=False)

# Create and train model
print("Creating volcanic CNN model...")
volcanic_model = create_volcanic_cnn_model(input_shape=(224, 224, 3), n_classes=5)
volcanic_model.summary()

print("\nTraining volcanic model...")
volcanic_history = train_model(volcanic_model, train_dataset, val_dataset, 'volcanic_cnn')

# Plot results
plot_training_history(volcanic_history, 'Volcanic CNN')

In [ ]:
# Load InSAR data
print("Loading InSAR deformation data...")
X_insar, y_insar = load_insar_data()

# Check data shape
print(f"InSAR data shape: {X_insar.shape}")
print(f"InSAR labels shape: {y_insar.shape}")

# Split data
split_idx = int(0.8 * len(X_insar))
X_train, X_val = X_insar[:split_idx], X_insar[split_idx:]
y_train, y_val = y_insar[:split_idx], y_insar[split_idx:]

# Create datasets
train_dataset = create_dataset(X_train, y_train, BATCH_SIZE)
val_dataset = create_dataset(X_val, y_val, BATCH_SIZE, shuffle=False)

# Create and train model
print("Creating InSAR CNN model...")
insar_model = create_insar_cnn_model(input_shape=(128, 128, 2), n_classes=2)
insar_model.summary()

print("\nTraining InSAR model...")
insar_history = train_model(insar_model, train_dataset, val_dataset, 'insar_cnn')

# Plot results
plot_training_history(insar_history, 'InSAR CNN')

## 8. Train InSAR Deformation Model

In [ ]:
# Create comprehensive fusion model combining all data sources
print("Creating comprehensive fusion model...")

# Prepare multi-modal dataset
def create_multimodal_dataset(X_seismic, X_thermal, X_insar, y, batch_size, shuffle=True):
    """
    Create dataset for multi-input model
    """
    # Ensure all labels are the same
    min_samples = min(len(X_seismic), len(X_thermal), len(X_insar))
    
    # Truncate to same length
    X_seismic = X_seismic[:min_samples]
    X_thermal = X_thermal[:min_samples]
    X_insar = X_insar[:min_samples]
    
    # Create unified labels (4 classes: safe, low, medium, high risk)
    y_unified = np.random.randint(0, 4, (min_samples,))  # In practice, combine all signals
    
    dataset = tf.data.Dataset.from_tensor_slices((
        {
            'seismic_input': X_seismic,
            'thermal_input': X_thermal,
            'insar_input': X_insar
        },
        y_unified
    ))
    
    if shuffle:
        dataset = dataset.shuffle(BUFFER_SIZE)
    
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset, y_unified

# Load all data
print("Loading all data sources for fusion model...")
X_seismic_fusion, _ = load_seismic_data()
X_thermal_fusion, _ = load_volcanic_thermal_data()
X_insar_fusion, _ = load_insar_data()

# Preprocess thermal data
X_thermal_fusion = tf.keras.applications.efficientnet.preprocess_input(X_thermal_fusion)

# Create train/val split
split_idx = int(0.8 * min(len(X_seismic_fusion), len(X_thermal_fusion), len(X_insar_fusion)))

# Create datasets
print("Creating multimodal datasets...")
train_dataset_fusion, y_train_fusion = create_multimodal_dataset(
    X_seismic_fusion[:split_idx],
    X_thermal_fusion[:split_idx],
    X_insar_fusion[:split_idx],
    None,
    BATCH_SIZE,
    shuffle=True
)

val_dataset_fusion, y_val_fusion = create_multimodal_dataset(
    X_seismic_fusion[split_idx:],
    X_thermal_fusion[split_idx:],
    X_insar_fusion[split_idx:],
    None,
    BATCH_SIZE,
    shuffle=False
)

# Create fusion model
fusion_model = create_comprehensive_fusion_model(
    seismic_shape=X_seismic_fusion.shape[1:],
    thermal_shape=(224, 224, 3),
    insar_shape=(128, 128, 2),
    n_classes=4
)

print("\nFusion Model Architecture:")
fusion_model.summary()

# Train fusion model
print("\nTraining comprehensive fusion model...")
fusion_history = train_model(fusion_model, train_dataset_fusion, val_dataset_fusion, 'fusion_model')

# Plot results
plot_training_history(fusion_history, 'Comprehensive Fusion Model')

## 9. Train Comprehensive Fusion Model

## 8. Fine-tuning (Optional)

In [ ]:
# Fine-tune the volcanic model by unfreezing base layers
def fine_tune_model(model, train_dataset, val_dataset, unfreeze_layers=20):
    """
    Fine-tune pre-trained layers
    """
    # Unfreeze the top layers of the base model
    base_model = model.layers[0]
    base_model.trainable = True
    
    # Freeze all layers except the last 'unfreeze_layers'
    for layer in base_model.layers[:-unfreeze_layers]:
        layer.trainable = False
    
    # Recompile with lower learning rate
    with strategy.scope():
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE/10),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3)]
        )
    
    # Fine-tune
    fine_tune_epochs = 20
    history = model.fit(
        train_dataset,
        epochs=fine_tune_epochs,
        validation_data=val_dataset,
        callbacks=[
            keras.callbacks.ModelCheckpoint(
                'volcanic_cnn_finetuned.h5',
                monitor='val_accuracy',
                save_best_only=True
            )
        ]
    )
    
    return history

# Uncomment to fine-tune
# fine_tune_history = fine_tune_model(volcanic_model, train_dataset, val_dataset)

## 9. Model Evaluation and Export

In [ ]:
def evaluate_model(model, test_dataset, model_name):
    """
    Comprehensive model evaluation
    """
    results = model.evaluate(test_dataset, verbose=1)
    print(f"\n{model_name} Test Results:")
    for metric, value in zip(model.metrics_names, results):
        print(f"{metric}: {value:.4f}")
    
    return results

def export_model_for_deployment(model, model_name):
    """
    Export model in multiple formats
    """
    # SavedModel format (recommended for TensorFlow Serving)
    model.save(f'{model_name}_saved_model')
    print(f"Saved model to {model_name}_saved_model/")
    
    # TFLite conversion for edge deployment
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    
    with open(f'{model_name}.tflite', 'wb') as f:
        f.write(tflite_model)
    print(f"Saved TFLite model to {model_name}.tflite")
    
    # ONNX export (requires tf2onnx)
    # !pip install tf2onnx
    # !python -m tf2onnx.convert --saved-model {model_name}_saved_model --output {model_name}.onnx

# Export all models
print("Exporting models for deployment...")
export_model_for_deployment(seismic_model, 'seismic_lstm')
export_model_for_deployment(volcanic_model, 'volcanic_cnn')
export_model_for_deployment(insar_model, 'insar_cnn')
export_model_for_deployment(fusion_model, 'fusion_model')

## 10. Inference Pipeline

In [ ]:
class PrometheusInference:
    """
    Unified inference pipeline for Prometheus models
    """
    def __init__(self, seismic_model_path, volcanic_model_path, insar_model_path=None, fusion_model_path=None):
        self.seismic_model = tf.keras.models.load_model(seismic_model_path)
        self.volcanic_model = tf.keras.models.load_model(volcanic_model_path)
        self.insar_model = tf.keras.models.load_model(insar_model_path) if insar_model_path else None
        self.fusion_model = tf.keras.models.load_model(fusion_model_path) if fusion_model_path else None
        
        # Alert thresholds
        self.seismic_threshold = 0.8
        self.volcanic_threshold = 0.7
        self.deformation_threshold = 0.75
    
    def predict_seismic_event(self, seismic_data):
        """
        Predict seismic event probability
        """
        predictions = self.seismic_model.predict(seismic_data)
        return predictions
    
    def predict_volcanic_activity(self, thermal_image):
        """
        Predict volcanic activity level
        """
        processed_image = tf.keras.applications.efficientnet.preprocess_input(thermal_image)
        predictions = self.volcanic_model.predict(processed_image)
        return predictions
    
    def predict_deformation(self, insar_data):
        """
        Predict ground deformation risk
        """
        if self.insar_model:
            predictions = self.insar_model.predict(insar_data)
            return predictions
        return None
    
    def predict_comprehensive_risk(self, seismic_data, thermal_image, insar_data):
        """
        Use fusion model for comprehensive risk assessment
        """
        if self.fusion_model:
            processed_thermal = tf.keras.applications.efficientnet.preprocess_input(thermal_image)
            predictions = self.fusion_model.predict({
                'seismic_input': seismic_data,
                'thermal_input': processed_thermal,
                'insar_input': insar_data
            })
            return predictions
        return None
    
    def generate_alert(self, seismic_prob, volcanic_prob, deformation_prob=None):
        """
        Generate alert based on predictions
        """
        alert_level = "GREEN"
        risk_factors = []
        
        if seismic_prob > self.seismic_threshold:
            risk_factors.append("High seismic activity")
        if volcanic_prob > self.volcanic_threshold:
            risk_factors.append("Elevated thermal anomaly")
        if deformation_prob and deformation_prob > self.deformation_threshold:
            risk_factors.append("Significant ground deformation")
        
        # Determine alert level
        if len(risk_factors) >= 3:
            alert_level = "RED"
        elif len(risk_factors) >= 2:
            alert_level = "ORANGE"
        elif len(risk_factors) >= 1:
            alert_level = "YELLOW"
        
        return {
            "alert_level": alert_level,
            "seismic_probability": float(seismic_prob),
            "volcanic_probability": float(volcanic_prob),
            "deformation_probability": float(deformation_prob) if deformation_prob else None,
            "risk_factors": risk_factors,
            "timestamp": pd.Timestamp.now().isoformat()
        }

# Example usage
# inference = PrometheusInference('seismic_lstm_best.h5', 'volcanic_cnn_best.h5', 'insar_cnn_best.h5', 'fusion_model_best.h5')
# alert = inference.generate_alert(0.85, 0.92, 0.78)
# print(json.dumps(alert, indent=2))

## 11. Cleanup and Save Results

In [ ]:
# Save training logs
import json
import datetime

training_summary = {
    "training_date": datetime.datetime.now().isoformat(),
    "tpu_available": tpu is not None,
    "num_replicas": strategy.num_replicas_in_sync,
    "models_trained": ["seismic_lstm", "volcanic_cnn", "insar_cnn", "fusion_model"],
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "data_sources": {
        "seismic": "USGS Earthquake API",
        "thermal": "Sentinel Hub (Sentinel-3 SLSTR)",
        "insar": "ASF HyP3 (Sentinel-1)"
    },
    "final_metrics": {
        "seismic_accuracy": float(seismic_history.history['val_accuracy'][-1]),
        "volcanic_accuracy": float(volcanic_history.history['val_accuracy'][-1]),
        "insar_accuracy": float(insar_history.history['val_accuracy'][-1]),
        "fusion_accuracy": float(fusion_history.history['val_accuracy'][-1])
    }
}

with open('training_summary.json', 'w') as f:
    json.dump(training_summary, f, indent=2)

print("\nTraining Summary:")
print(json.dumps(training_summary, indent=2))

# List all saved artifacts
print("\nSaved artifacts:")
!ls -la *.h5 *.tflite *.json *.png *_saved_model/